# Load Libraries

In [1]:
import os
import warnings
import logging
import sys
import pickle
import numpy as np
import pandas as pd
import geopandas as gpd
import dotenv
import pyet
import matplotlib.pyplot as plt


# Load environment variables from .env file
dotenv.load_dotenv()

# Set up logging
logging.basicConfig(level=logging.INFO)

# Suppress warnings
warnings.filterwarnings("ignore")

# Set display options for pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_colwidth', None)

# Load Data

In [2]:
# Load Parquet Data
data = pd.read_parquet('../../data/Iran_Monthly_ETo_1951_2025.parquet')
logging.info("Data loaded from parquet file.")    

INFO:root:Data loaded from parquet file.


In [3]:
data

,year,month,region_id,region_name,station_id,station_name,lat,lon,station_elevation,tmax,tmax_count,tmin,tmin_count,tm,tm_count,umax,umax_count,umin,umin_count,um,um_count,ffm,ffm_count,sshn,sshn_count,rrr24,rrr24_count,date,FAO56,Hargreaves,Blaney_Criddle,Oudin
0,1951,1,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,1951-01-01,NaN,NaN,NaN,NaN
1,1951,2,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,1951-02-01,NaN,NaN,NaN,NaN
2,1951,3,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,1951-03-01,NaN,NaN,NaN,NaN
3,1951,4,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,1951-04-01,NaN,NaN,NaN,NaN
4,1951,5,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,1951-05-01,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
694273,2025,5,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,24.46,13,12.53,10,19.07,10,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,12.20,11,2025-05-01,NaN,147.74,110.63,121.41
694274,2025,6,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,32.03,30,12.43,28,22.12,28,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,29,2025-06-01,NaN,207.46,125.91,138.41
694275,2025,7,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,36.54,31,17.80,30,27.14,30,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,3.10,30,2025-07-01,NaN,232.24,149.49,166.81
694276,2025,8,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,37.20,20,19.91,11,29.04,11,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,19,2025-08-01,NaN,213.22,145.58,162.03


# Data Description

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 694278 entries, 0 to 694277
Data columns (total 32 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   year               694278 non-null  int32         
 1   month              694278 non-null  int32         
 2   region_id          694278 non-null  object        
 3   region_name        694278 non-null  object        
 4   station_id         694278 non-null  object        
 5   station_name       694278 non-null  object        
 6   lat                694278 non-null  float64       
 7   lon                694278 non-null  float64       
 8   station_elevation  694278 non-null  float64       
 9   tmax               162557 non-null  float64       
 10  tmax_count         694278 non-null  int64         
 11  tmin               161808 non-null  float64       
 12  tmin_count         694278 non-null  int64         
 13  tm                 161595 non-null  float64 

In [5]:
data = data[[
    'region_id', 'region_name', 'station_id', 'station_name',
    'lat', 'lon', 'station_elevation',
    'date', 'year', 'month',
    'tmax', 'tmax_count', 'tmin', 'tmin_count', 'tm', 'tm_count', 
    'umax', 'umax_count', 'umin', 'umin_count', 'um', 'um_count',
    'ffm', 'ffm_count', 'sshn', 'sshn_count', 'rrr24', 'rrr24_count',
    'FAO56', 'Hargreaves', 'Blaney_Criddle', 'Oudin'
]]

data

,region_id,region_name,station_id,station_name,lat,lon,station_elevation,date,year,month,tmax,tmax_count,tmin,tmin_count,tm,tm_count,umax,umax_count,umin,umin_count,um,um_count,ffm,ffm_count,sshn,sshn_count,rrr24,rrr24_count,FAO56,Hargreaves,Blaney_Criddle,Oudin
0,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-01-01,1951,1,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN
1,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-02-01,1951,2,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN
2,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-03-01,1951,3,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN
3,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-04-01,1951,4,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN
4,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-05-01,1951,5,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
694273,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-05-01,2025,5,24.46,13,12.53,10,19.07,10,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,12.20,11,NaN,147.74,110.63,121.41
694274,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-06-01,2025,6,32.03,30,12.43,28,22.12,28,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,29,NaN,207.46,125.91,138.41
694275,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-07-01,2025,7,36.54,31,17.80,30,27.14,30,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,3.10,30,NaN,232.24,149.49,166.81
694276,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-08-01,2025,8,37.20,20,19.91,11,29.04,11,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,19,NaN,213.22,145.58,162.03


In [6]:
print(f"Number of unique regions: {data.region_name.nunique()}")
print(f"Region names: {data.region_name.unique().tolist()}")

Number of unique regions: 32
Region names: ['Airforce', 'Alborz', 'Ardebil', 'Azarbayjan-E-Gharbi', 'Azarbayjan-E-Sharghi', 'Bushehr', 'Chaharmahal Va Bakhtiari', 'Esfahan', 'Fars', 'Gilan', 'Golestan', 'Hamedan', 'Hormozgan', 'Ilam', 'Kerman', 'Kermanshah', 'Khohgiluyeh Va Boyerahmad', 'Khorasan Razavi', 'Khuzestan', 'Kordestan', 'Lorestan', 'Markazi', 'Mazandaran', 'North Khorasan', 'Qazvin', 'Qom', 'Semnan', 'Sistan Va Baluchestan', 'South Khorasan', 'Tehran', 'Yazd', 'Zanjan']


In [8]:
d = data[['region_id', 'region_name', 'station_name', 'station_id', 'lat', 'lon', 'station_elevation']].drop_duplicates().reset_index(drop=True)
d[d.duplicated(subset=['region_name', 'station_name'], keep=False)]

,region_id,region_name,station_name,station_id,lat,lon,station_elevation
359,OIKK,Kerman,Golbaf,19680,29.85,57.73,1665.00
360,OIKK,Kerman,Golbaf,99583,29.85,57.73,1665.00
519,OICK,Lorestan,Rymaleh,19124,33.64,48.40,1650.00
520,OICK,Lorestan,Rymaleh,99471,33.64,48.40,1650.00
556,MASA,Mazandaran,Nowshahr,24002,36.95,51.66,9999.00
557,MASA,Mazandaran,Nowshahr,40734,36.66,51.47,-20.90
583,OIMN,North Khorasan,Shirvan,18203,37.38,57.92,1187.00
584,OIMN,North Khorasan,Shirvan,99270,37.43,57.83,1051.00
631,OIIS,Semnan,Shahmirzad,90250,35.78,53.37,1960.00
632,OIIS,Semnan,Shahmirzad,99386,35.77,53.35,1969.00


In [13]:
print(f"Number of unique stations:")
data[['region_name', 'station_name']].drop_duplicates().reset_index(drop=True)

Number of unique stations:


,region_name,station_name
0,Airforce,Dowshan Tappeh
1,Airforce,Hamedan (Nozheh)
2,Airforce,Khurbirjand
3,Airforce,Konarak (Airport)
4,Alborz,Asara
...,...,...
763,Zanjan,Soltaniyeh
764,Zanjan,Zanjan
765,Zanjan,Zanjan (Airport)
766,Zanjan,Zarinabad(Egrood)
